# 04 — Reflective / learning agent

**Definition:** produce -> **critique** -> revise, until the critic is satisfied (or I run
out of patience). Covers basic **Reflection**, **Reflexion**, and **Self-Correction**.

```
      Task
        |
        v
   +----------+
+->| GENERATE |  produce a draft
|  +----------+
|        |
|        v
|  +----------+
|  | REFLECT  |  critique it (a DIFFERENT LLM role)
|  +----------+
|        |
+--------+ "not good enough"
         |
         +------ "good enough" ------> Final
```

The critique is **fed back as input** to the generator. That's the whole trick.

**Three flavours**, and the difference is entirely *how grounded the critic is*:

| flavour | critic uses | feedback is | good for |
|---|---|---|---|
| Basic reflection | LLM judgement only | prose critique | writing, essays, style |
| Reflexion | LLM + external tools (search, tests) | grounded critique | research, factual claims |
| Self-correction | a **deterministic** checker (compiler, tests, linter) | pass/fail + error text | code, SQL, JSON |

The stronger the grounding, the more reflection actually helps. A critic that is just
"another LLM opinion" plateaus after about two rounds. I build all three below.

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

# Part A — basic reflection

## Two prompts, two opposing personas

If both roles are "a helpful assistant", the critique comes back toothless. The critic has
to be given a job that is explicitly adversarial.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# The GENERATOR: writes, and revises when handed a critique.
generate_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an essay assistant tasked with writing excellent 3-paragraph essays.\n"
            "Generate the best essay possible for the user's request.\n"
            "If the user provides a critique, respond with a REVISED version of your "
            "previous attempt that addresses every point raised.",
        ),
        MessagesPlaceholder(variable_name="messages"),   # the accumulating history
    ]
)

# The REFLECTOR: a deliberately harsh, specific critic.
reflect_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a demanding teacher grading an essay submission.\n"
            "Generate a critique and concrete recommendations.\n"
            "Be SPECIFIC: comment on length, depth, style, and factual accuracy.\n"
            "Give actionable instructions, not vague praise. Keep it under 200 words.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

generate_chain = generate_prompt | llm
reflect_chain = reflect_prompt | llm

## The role flip — the one line that confused me longest

```python
cls_map = {"ai": HumanMessage, "human": AIMessage}
```

The generator and the reflector are the **same model** playing **opposite roles**. From the
reflector's point of view the roles are inverted:

| message | generator sees it as | reflector must see it as |
|---|---|---|
| the original task | Human (input) | its own instruction context |
| the draft essay | **AI** (its own output) | **Human** (the thing to review) |
| the critique | **Human** (feedback to act on) | **AI** (its own output) |

If I skip the flip, the reflector reads the draft as an `AIMessage` — as *its own work* —
and mostly agrees with itself. Relabelling the draft as a `HumanMessage` turns it into
"here is someone's work, review it". That's what makes the critic actually critical.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph


def generation_node(state: MessagesState) -> dict:
    """Write a draft, or revise using the critique sitting in the message list."""
    return {"messages": [generate_chain.invoke({"messages": state["messages"]})]}


def reflection_node(state: MessagesState) -> dict:
    """Critique the latest draft. Note the role inversion."""
    cls_map = {"ai": HumanMessage, "human": AIMessage}

    # Keep the ORIGINAL request as-is (index 0) so the critic knows the task,
    # then flip the role of everything after it.
    translated = [state["messages"][0]] + [
        cls_map[msg.type](content=msg.content) for msg in state["messages"][1:]
    ]

    res = reflect_chain.invoke({"messages": translated})

    # Return the critique as a HumanMessage, NOT an AIMessage:
    # the generator has to read it as instructions from a user.
    return {"messages": [HumanMessage(content=res.content)]}

## Termination

The message list grows by 2 per round (draft + critique), starting from 1 (the request):

```
request     1
draft 1     2
critique 1  3
draft 2     4
critique 2  5
draft 3     6   <- stop here: 2 revisions done
```

**Why a hard cap is mandatory:** LLM critics are biased toward finding faults. Ask "what's
wrong with this?" and they will always find something, forever. The critic will not stop on
its own — nothing in basic reflection ever says "approved".

In [ ]:
MAX_MESSAGES = 6      # ~3 drafts + 2 critiques


def should_continue(state: MessagesState) -> str:
    if len(state["messages"]) > MAX_MESSAGES:
        return END
    return "reflect"


builder = StateGraph(MessagesState)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)

builder.add_edge(START, "generate")
builder.add_conditional_edges("generate", should_continue, ["reflect", END])
builder.add_edge("reflect", "generate")     # the reflection loop

reflect_graph = builder.compile()
show(reflect_graph)

## Run it and watch the drafts change

In [ ]:
request = HumanMessage(
    content=(
        "Write a short essay on why event-driven architecture beats "
        "request-response for high-throughput systems."
    )
)

for chunk in reflect_graph.stream({"messages": [request]}, stream_mode="updates"):
    for node, update in chunk.items():
        content = update["messages"][-1].content
        tag = "DRAFT" if node == "generate" else "CRITIQUE"
        print(f"\n{'=' * 60}\n{tag} ({node})\n{'=' * 60}")
        print(content[:600] + ("..." if len(content) > 600 else ""))

# Part B — scored reflection (a real stop signal)

Prose critiques give me nothing to branch on. Forcing a **score** makes the loop
deterministic: stop when `score >= threshold`. This is the version I'd actually ship.

In [ ]:
from typing import List, TypedDict

from pydantic import BaseModel, Field


class Critique(BaseModel):
    """A structured, scored critique - makes 'good enough' a threshold, not a vibe."""

    score: int = Field(description="Quality score from 1 (poor) to 10 (excellent)")
    strengths: List[str] = Field(description="What works well")
    issues: List[str] = Field(description="Concrete problems that must be fixed")
    approved: bool = Field(description="True only if score >= 8 and no critical issues")


class ScoredState(TypedDict):
    task: str
    draft: str
    critique: str
    score: int
    iteration: int


critic_llm = llm.with_structured_output(Critique, method="json_schema")

In [ ]:
def gen_node(state: ScoredState) -> dict:
    """Draft on iteration 0; revise thereafter using the stored critique."""
    if state.get("draft"):
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your previous draft:\n{state['draft']}\n\n"
            f"Critique to address:\n{state['critique']}\n\n"
            "Write an improved version fixing every issue listed."
        )
    else:
        prompt = f"Task: {state['task']}\n\nWrite the best possible response."

    draft = llm.invoke(prompt).content
    it = state.get("iteration", 0) + 1
    print(f"\nDraft #{it} ({len(draft)} chars)")
    return {"draft": draft, "iteration": it}


def critic_node(state: ScoredState) -> dict:
    """Score the draft against an EXPLICIT RUBRIC - grounding beats vibes."""
    c = critic_llm.invoke(
        f"Task: {state['task']}\n\nSubmission:\n{state['draft']}\n\n"
        "Grade it on accuracy, depth, clarity and structure.\n"
        "You are a HARSH grader. Reserve 9-10 for publication-ready work with no "
        "hand-waving. Any vagueness, missing caveat or unexplained term caps the score at 7."
    )
    print(f"Score: {c.score}/10 | approved={c.approved}")
    for issue in c.issues:
        print(f"   - {issue}")

    return {
        "score": c.score,
        "critique": "Issues to fix:\n" + "\n".join(f"- {i}" for i in c.issues),
    }

### The router needs *two* exits

Score-only can loop forever (the bar may never be met); iteration-only can ship garbage.
I need both.

**Calibration note.** My first attempt used `score >= 8` with a plain "grade this strictly"
prompt. It scored the very first draft 9/10 and exited without ever looping — so the demo
proved nothing, and worse, a bar the first draft always clears is a bar that isn't doing
any work. The grader prompt above is now explicitly harsh and the threshold is 9. Picking
the threshold *with* the rubric, not independently of it, is the actual lesson.

In [ ]:
MAX_ITERS = 3
QUALITY_BAR = 9


def route(state: ScoredState) -> str:
    if state["score"] >= QUALITY_BAR:
        print("approved - score threshold met")
        return END
    if state["iteration"] >= MAX_ITERS:
        print("max iterations reached - shipping the current draft")
        return END
    return "generate"


sb = StateGraph(ScoredState)
sb.add_node("generate", gen_node)
sb.add_node("critic", critic_node)

sb.add_edge(START, "generate")
sb.add_edge("generate", "critic")
sb.add_conditional_edges("critic", route, ["generate", END])

scored_graph = sb.compile()

out = scored_graph.invoke(
    {"task": "Explain the CAP theorem to a senior backend engineer in under 200 words.", "iteration": 0}
)
print("\n" + "=" * 60)
print(f"FINAL (score {out['score']}/10, {out['iteration']} iteration(s))\n")
print(out["draft"])

# Part C — self-correction (a critic that cannot be sweet-talked)

The strongest form. The critic isn't an LLM opinion, it's a **program**: the code either
runs or it doesn't. No sycophancy is possible, and the error text fed back is precise.

In [ ]:
class CodeState(TypedDict):
    task: str
    code: str
    error: str
    attempts: int


def write_code(state: CodeState) -> dict:
    """Generate code; on a retry, include the ACTUAL error text."""
    if state.get("error"):
        prompt = (
            f"Task: {state['task']}\n\n"
            f"Your code:\n{state['code']}\n\n"
            f"It FAILED with:\n{state['error']}\n\n"
            "Return corrected Python code only. No markdown fences, no explanation."
        )
    else:
        prompt = f"{state['task']}\n\nReturn Python code only. No markdown fences, no explanation."

    code = llm.invoke(prompt).content.strip()
    # Strip fences defensively - models add them even when told not to.
    code = code.removeprefix("```python").removeprefix("```").removesuffix("```").strip()
    return {"code": code, "attempts": state.get("attempts", 0) + 1}


def run_tests(state: CodeState) -> dict:
    """The DETERMINISTIC critic. Truth, not opinion."""
    try:
        exec(state["code"], {})     # demo only - sandbox this for anything real
        print(f"   attempt {state['attempts']}: tests passed")
        return {"error": ""}
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        print(f"   attempt {state['attempts']}: {err}")
        return {"error": err}       # this exact string is fed back to the LLM


def code_route(state: CodeState) -> str:
    if not state["error"]:
        return END                  # passed
    if state["attempts"] >= 3:
        return END                  # give up gracefully
    return "write"                  # retry with the error in context

In [ ]:
cb = StateGraph(CodeState)
cb.add_node("write", write_code)
cb.add_node("test", run_tests)
cb.add_edge(START, "write")
cb.add_edge("write", "test")
cb.add_conditional_edges("test", code_route, ["write", END])
code_graph = cb.compile()

res = code_graph.invoke(
    {
        "task": (
            "Write a function `fib(n)` returning the nth Fibonacci number, "
            "then assert fib(10) == 55 and assert fib(0) == 0."
        ),
        "attempts": 0,
    }
)
print("\n--- FINAL CODE ---\n" + res["code"])

### Proving the loop actually repairs things

That passed on attempt 1, which proves the graph runs but not that reflection *does*
anything. I tried a couple of "tricky" tasks to force a failure and the model solved those
first try too — writing a prompt the model reliably fails is surprisingly hard.

So instead of contriving a hard task, I plant a bug and start the graph at the **test**
node. Same two nodes, same router, just a different entry point — now the failure is
guaranteed and I can watch the repair.

In [ ]:
rb = StateGraph(CodeState)
rb.add_node("write", write_code)
rb.add_node("test", run_tests)
rb.add_edge(START, "test")                 # start by TESTING code I planted
rb.add_edge("write", "test")
rb.add_conditional_edges("test", code_route, ["write", END])
repair_graph = rb.compile()

BROKEN = """
def median(xs):
    xs = sorted(xs)
    return xs[len(xs) // 2]        # wrong: no averaging for even-length lists

assert median([1, 3, 2, 4]) == 2.5, "even-length list must average the two middle values"
assert median([5]) == 5, "odd-length list must return the middle value"
"""
# Assertion messages matter here: that string IS the critique the LLM gets back.
# A bare `assert x == y` gives it "AssertionError:" and nothing to work with.

res2 = repair_graph.invoke(
    {
        "task": (
            "Fix `median(xs)`. For an even-length list it must return the average of the "
            "two middle values. Keep the asserts."
        ),
        "code": BROKEN,
        "attempts": 0,
    }
)
print(f"\nattempts used: {res2['attempts']}, final error: {res2['error'] or 'none'}")
print("\n--- FINAL CODE ---\n" + res2["code"])

## Notes to self

**Doesn't reflection just make the model agree with itself?** Yes — *if the critic isn't
grounded*. Mitigations, strongest first:

1. give the critic **tools** (run the test, look the fact up) -> Reflexion / self-correction
2. give the critic a **rubric** (explicit, scored criteria)
3. use a **different, stronger model** as the critic
4. force **structured output with a numeric score**, so "good enough" is a threshold

**Reflector vs the replanner in 03:** the replanner judges the *path* ("what's left?"), the
reflector judges the *output* ("is this any good?"). Orthogonal; real systems have both.

**Failure modes I've hit:**

| symptom | cause | fix |
|---|---|---|
| critic always approves | same persona for both roles | harsh critic persona + the role flip |
| never approves | LLM bias toward finding faults | hard iteration cap |
| quality plateaus at round 2 | ungrounded critic | give it tools / tests / a rubric |
| revisions get worse | critique too vague | force structured `issues: List[str]` |
| context explodes | full history every round | keep only the latest draft + critique |

**Cost:** ~2x LLM calls per round. Worth it when quality matters more than latency
(writing, code, analysis, SQL); wrong choice when the critic can't be grounded.

**API I used:**

```python
MessagesPlaceholder(variable_name="messages")        # history slot in a prompt template
{"ai": HumanMessage, "human": AIMessage}[msg.type]   # the role flip
llm.with_structured_output(Critique, method="json_schema")
if state["iteration"] >= MAX: return END             # the cap - never optional
exec(...) / run a test suite                         # the deterministic critic
```

Next: **05 — Memory**, where the agent stops forgetting me between runs.